# 11장 · OpenAI Responses API의 요청·응답 구조

SDK는 Python에서 요청을 만들고 응답을 읽게 해 주는 라이브러리입니다. 실제 응답에서 메시지, 출력 본문, 사용량 필드를 따로 살펴볼 수 있습니다.

Responses API 요청에서 질문의 역할과 내용을 지정하고, 수업용 응답에서 답변·완료 상태·사용량을 각각 읽습니다. 응답 일부만 받았을 때는 완료 상태가 해석 기준이 됩니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 httpx 0.28.1, openai 3.35.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 사용자 메시지의 role과 content

SDK는 파이썬 코드에서 서비스의 요청과 응답을 다루게 해 주는 라이브러리입니다. 먼저 보내려는 내용만 조립하면 실제 통신 없이도 모델 이름과 사용자 메시지가 어느 칸에 들어 있는지 볼 수 있습니다.


### API·SDK·토큰의 관계

<p>API는 프로그램이 서비스와 요청·응답을 주고받는 규약입니다. Python SDK는 그 규약을 Python 객체와 메서드로 사용하도록 돕는 라이브러리입니다. SDK를 설치하는 것과 서버에 모델 요청을 보내는 것은 별개의 일입니다.</p>

<p>모델은 요청의 입력을 처리하고 출력을 생성합니다. 토큰은 모델이 텍스트 등을 처리할 때 사용하는 단위이며 한국어 글자 수와 일치하지 않습니다. 사용량 필드의 입력 토큰·출력 토큰과 답변 글자 수를 같은 값으로 취급하지 않습니다.</p>

<p>컨텍스트는 모델이 이번 요청에서 참고할 입력 범위입니다. 긴 문서를 넣는다고 전체 내용을 정확히 기억한다는 뜻은 아닙니다. 모델 선택에는 기능 지원, 입력 길이, 응답 품질, 지연, 비용 조건을 함께 확인해야 합니다. 이번 교재는 특정 모델을 모든 과제에 최선이라고 정하지 않습니다.</p>

<table><caption>이번 예제의 책임</caption><thead><tr><th scope="col">요소</th><th scope="col">하는 일</th></tr>
</thead><tbody><tr><td>내 Python 코드</td><td>입력 구성·호출·출력 기록</td></tr>
<tr><td>OpenAI SDK</td><td>요청 직렬화·응답 객체 처리</td></tr>
<tr><td>실제 API 서버</td><td>선택 모델 실행</td></tr>
<tr><td>실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.</td><td>실제 서비스에 요청하고 응답 읽기</td></tr>
</tbody></table>


### 호출 전 입력 검증

<p>실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.</p>

### 셀 01 · role과 content 구성

model은 계정에서 사용할 실제 모델의 이름입니다. 실제 계정에서 선택할 모델과 구분하세요.


In [ ]:
import os
from openai import OpenAI

client = OpenAI(timeout=45, max_retries=1)
model_name = os.getenv("OPENAI_MODEL") or "gpt-4o-mini"
question = "SDK가 하는 일을 한 문장으로 설명하세요."
request = {"model": model_name, "input": [{"role": "user", "content": question}], "max_output_tokens": 300}
print(request["input"][0]["content"])

SDK가 하는 일을 한 문장으로 설명하세요.


원래 질문이 그대로 나옵니다. max_output_tokens는 생성할 출력의 한도이고 입력 문장 길이나 실제 생성 토큰 수가 아닙니다.

이 셀을 여러 번 실행해도 호출 비용이 쌓이지 않습니다. 만든 것은 요청 내용을 담은 파이썬 사전이고 마지막 줄은 그 안의 질문을 읽습니다. 다음 단계에서 응답을 볼 때는 어떤 전송 함수가 그 값을 돌려줬는지도 확인하세요.


### 대화 메시지 목록 구성

<p>이 예제의 input에는 사용자 메시지 하나가 있습니다. 프로그램을 다시 실행해도 지난 요청의 내용이 자동으로 포함된다고 가정하지 않습니다. 후속 질문에 앞선 내용이 필요하면 그 내용을 입력으로 전달하는 방식을 명시적으로 구현해야 합니다.</p>


## OpenAI Responses API 호출 · 실제 모델 호출

실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.

### 셀 02 · 요청 모델과 출력 길이 확인

제공 함수는 요청에서 user 역할을 확인하고, message 하나와 사용량 예시가 든 HTTP 응답을 준비합니다.

제공 함수는 네트워크 대신 예제 응답을 만듭니다. 셀 03에서는 본문을, 과제 04에서는 같은 응답의 사용량을 읽습니다. 처음에는 함수 안의 생성 코드보다 반환된 응답 한 건을 따라가세요.


In [ ]:
print(request["model"], request["max_output_tokens"])

gpt-4o-mini 300


이 셀은 함수를 정의합니다. 아직 요청을 보내지 않았으므로 답변 출력이 없습니다.


#### 가상 응답으로 확인할 수 있는 범위

<mark>MockTransport는 요청을 받아 준비한 응답을 돌려줍니다.</mark> SDK가 어떤 메시지를 보냈고 응답에서 어느 필드를 읽었는지 추적하세요. 문장이 자연스럽게 출력됐다는 사실만으로 모델 서버의 응답 시간이나 생성 품질을 평가할 수는 없습니다.

선수 개념 · 6장 HTTP 요청과 JSON 응답 처리

#### 응답 조각이 있어도 완료 상태를 먼저 확인하기

```mermaid
flowchart TD
nreply["응답 받기"]
  nstatus["status 검사"]
  nshow["본문 표시"]
  nwait["미완료 안내"]
  nreply --> nstatus
  nstatus --> nshow
  nstatus --> nwait
```

partial의 text에는 ‘가상 안’이 남아 있습니다. 상태가 incomplete인 이 실행에서는 미완료 안내가 출력됩니다.

### SDK와 API 키 준비 · 실제 모델 호출

<p>이 호출은 SDK가 요청을 구성하고 응답을 읽는지 확인합니다. 실제 모델의 답변 품질·사용 가능 모델·계정 권한·요금·네트워크는 확인하지 않습니다.</p>


### 셀 03 · output_text 읽기

응답 객체의 output_text에서 실제 반환 문장을 읽으세요.


In [ ]:
response = client.responses.create(**request)
print(response.output_text)

SDK(소프트웨어 개발 키트)는 개발자가 특정 플랫폼이나 애플리케이션을 만들고 통합할 수 있도록 도구, 라이브러리, 문서 등을 제공하는 패키지입니다.


SDK가 꺼낸 본문은 ‘가상 안내를 확인했습니다.’입니다. 이 문장은 제공 함수의 응답 본문에서 왔습니다. 실제 서비스 모델의 생성 결과나 품질을 보여 주지는 않습니다.


### SDK 요청 함수

<p>셀 03에서 request 앞에 붙인 별표 두 개는 셀 01에서 만든 사전을 responses.create의 이름 있는 인자로 풀어 전달합니다. 이 요청에는 model, input, max_output_tokens가 들어 있습니다. 한도를 300으로 두어도 응답의 실제 총량은 usage에서 별도로 읽습니다.</p>

<p>SDK는 API의 HTTP 응답을 응답 객체로 변환합니다. 이를 받은 SDK가 response 객체를 만들고, response.output_text가 그 안의 텍스트를 꺼냅니다. 상태와 사용량은 각각 response.status와 response.usage에서 읽을 수 있습니다.</p>

## usage와 응답 완료 상태

토큰은 모델이 입력과 출력을 세는 단위입니다. 과금이나 지연을 추측하지 않고 응답에 실제 제공된 값을 읽습니다. 요청 상태가 미완료면 본문 길이만 보고 성공으로 기록하지 않습니다.


### 과제 04 · usage의 토큰 합계

제공 응답과 같은 사용량 사전입니다. 사용량이 보고된 경우 총 토큰 수를 출력하세요.


In [ ]:
usage = response.usage.model_dump()
# 응답의 총량을 꺼내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>usage = response.usage.model_dump()
print(usage[&quot;total_tokens&quot;])</code></pre>
<p>실행 결과</p><pre><code>18</code></pre>
<p>18은 제공 응답에 넣어 둔 수치입니다. 실제 요청의 경우에는 모델·입력마다 달라지며 사용량이 빠진 경우도 따로 남깁니다.</p>
</details>


### 응답 객체의 본문과 사용량

<p>Responses API의 output에는 텍스트 외의 항목도 들어갈 수 있습니다. 첫 번째 항목의 첫 번째 필드가 항상 답변이라고 가정하지 않고 SDK의 output_text를 사용합니다. status와 usage도 따로 확인합니다.</p>

<p>실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.</p>

<table><caption>결과 필드</caption><thead><tr><th scope="col">필드</th><th scope="col">해석</th></tr>
</thead><tbody><tr><td>mode</td><td>mock 또는 live 실행 구분</td></tr>
<tr><td>status</td><td>응답의 완료 상태</td></tr>
<tr><td>answer</td><td>SDK가 모은 텍스트 출력</td></tr>
<tr><td>usage</td><td>서버 응답의 토큰 사용량 또는 없음</td></tr>
<tr><td>seconds</td><td>로컬에서 측정한 요청 대기 포함 시간</td></tr>
</tbody></table>


### 셀 05 · 미완료 상태 판별

통신은 끝났지만 서비스가 완료하지 못했다고 가정합니다. 본문을 화면에 내보내도 될까요?


In [5]:
partial = {"status": "incomplete", "text": "가상 안"}
if partial["status"] == "completed":
    print(partial["text"])
else:
    print("미완료 응답")

미완료 응답


‘미완료 응답’이 남습니다. 실제 호출 기록에는 모델, 상태, 사용량과 경과 시간을 구분해 두고 오류가 나면 키나 요청 원문 전체 대신 필요한 오류 종류를 기록합니다.


### 완료 상태를 확인한 뒤 본문 읽기

<p>예제의 partial에는 짧은 text가 있지만 status는 incomplete입니다. 미리 받은 글자가 있어도 완성된 답변으로 표시하기 전에는 상태를 확인해야 합니다. 이 셀의 출력은 ‘미완료 응답’입니다.</p>


### 과제 06 · 응답 상태에 따라 표시할 본문 선택

셀 05의 partial에는 text가 있지만 응답은 아직 끝나지 않았습니다. 완료 응답일 때만 본문을 반환하는 display_text를 작성하세요. partial에 적용하면 ‘응답 대기’가 나와야 합니다.

In [ ]:
def display_text(reply):
    # 완료된 응답만 본문을 표시하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def display_text(reply):
    if reply[&quot;status&quot;] == &quot;completed&quot;:
        return reply[&quot;text&quot;]
    return &quot;응답 대기&quot;

print(display_text(partial))</code></pre>
<p>실행 결과</p><pre><code>응답 대기</code></pre>
<p>text 값이 있는지만 검사하면 미완료인 ‘가상 안’까지 표시합니다. 같은 사전에서 status만 completed로 바꾸면 함수가 본문을 돌려주는지도 확인하세요.</p>
</details>

## 후속 질문이 어느 응답을 이어받는지 확인하기

실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.

### 셀 07 · 응답의 ID와 요청 본문을 함께 읽기

검수용 서버는 요청을 한 건 받을 때마다 다른 응답 ID를 붙입니다. 첫 질문을 보내고 SDK 응답 객체의 ID와 서버가 받은 질문을 확인하세요.

In [ ]:
first_turn = client.responses.create(**request)
print(first_turn.id, first_turn.output_text)

첫 응답의 id를 꺼내 후속 요청에 전달합니다. 출력 문장과 별개로 이 ID를 보관해 두면 다음 요청에 어느 대화를 잇는지 명시할 수 있습니다.

### 과제 08 · 첫 응답에 후속 질문 연결하기

첫 요청의 설정은 유지하고 input만 새 사용자 메시지로 바꾸세요. previous_response_id에 first_turn.id를 넣어 한 번 보내고 서버가 받은 이전 ID와 새 질문을 출력하세요.

In [ ]:
followup_input = [{"role": "user", "content": "한 가지 예를 들어 주세요."}]
# 새 요청을 만들어 client.responses.create에 넘기세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>followup_input = [{&quot;role&quot;: &quot;user&quot;, &quot;content&quot;: &quot;한 가지 예를 들어 주세요.&quot;}]
next_request = {**request, &quot;input&quot;: followup_input, &quot;previous_response_id&quot;: first_turn.id}
next_turn = client.responses.create(**next_request)
sent = next_request
print(sent[&quot;previous_response_id&quot;], sent[&quot;input&quot;][0][&quot;content&quot;])</code></pre>
<p>실행 결과</p><pre><code><첫 응답 ID> 한 가지 예를 들어 주세요.</code></pre>
<p>실제 응답의 ID와 usage를 읽고 후속 요청에 previous_response_id를 전달하세요.</p>
</details>

<mark>후속 질문을 보낼 때 필요한 연결 값은 앞 응답 객체의 id입니다.</mark> 앞 답변의 본문을 previous_response_id에 넣으면 서버가 응답을 식별할 수 없습니다. 이 연습에서는 기록한 요청을 읽어 ID를 넣은 자리와 새 질문이 들어간 자리를 구분할 수 있습니다.

실서비스에서 대화를 이어갈 때는 화면에 보여 줄 답변과 다음 요청에 필요한 응답 ID를 따로 보관하세요.

## 돌아보기

질문 하나의 입력, 응답 본문, 사용량을 다시 짚어 보세요. 같은 글자를 요청해도 라이브 응답의 값은 달라질 수 있습니다. 다음 장에서는 완료 응답을 받았을 때도 그 안의 공지 형식과 내용을 다시 확인합니다.
